# Fitting

In [1]:
import sys
sys.path.insert(0, '../../src/')

import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import tensorflow as tf

from qiskit.quantum_info import Operator
from tqdm.notebook import tqdm

from kraus_channels import KrausMap, isomery_to_kraus
from loss_functions import ProbabilityMSE, ProbabilityRValue
from optimization import ModelSPAM, ModelQuantumMap, Logger, model_saver
from quantum_channel import channel_fidelity
from experimental import counts_to_probs, generate_pauliInput_circuits, generate_pauli_circuits, marginalize_counts
from spam import SPAM, InitialState, CorruptionMatrix
from utils import saver, loader
from quantum_circuits import pqc_basic
from spectrum import channel_spectrum, complex_spacing_ratio
from quantum_circuits import integrable_circuit, nonintegrable_circuit


#np.set_printoptions(threshold=sys.maxsize)
np.set_printoptions(precision=4)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

2025-11-21 13:33:27.840034: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-21 13:33:28.277173: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-21 13:33:29.788797: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/tmp/ipykernel_45316/3028913883.py:9: DeprecationWarning: Using Qiskit with Pyth

In [2]:
def load_data(filename, n, seed, L):
    with open(filename, 'rb') as f:
        data = pkl.load(f)


    data = marginalize_counts(data, 0)

    targets = counts_to_probs(data)
    targets_spam = targets[:6**n]
    targets_map = targets[6**n:]

    np.random.seed(seed)
    
    circuit_target = integrable_circuit(n+1, L)
    unitary = Operator(circuit_target).data

    inputs_spam, _ = generate_pauliInput_circuits(n)
        
    inputs_map, circuit_list_map = (
                generate_pauli_circuits(n, None, N=5000-6**n)
        )
    
    return inputs_spam, targets_spam, inputs_map, targets_map, unitary

def fit_spam(inputs, 
             targets,
             num_iter = 3000,
             verbose = False):
    d = targets.shape[1]
    spam_model = SPAM(init = InitialState(d),
                    povm = CorruptionMatrix(d),
                    #povm = POVM(d),
                    )

    spam_opt = ModelSPAM(spam_model, tf.keras.optimizers.Adam(learning_rate=0.01))
        
    spam_opt.pretrain(100, verbose=False)

    spam_opt.train(inputs = inputs,
                    targets = targets,
                    num_iter = num_iter,
                    verbose = verbose,
                )
    
    return spam_model
    

def fit_model(inputs, 
              targets, 
              spam_model,
              num_iter = 3000,
              verbose=False):
    d = targets.shape[1]
    model = ModelQuantumMap(channel = KrausMap(d = d, 
                                        rank = d**2,
                                        spam = spam_model,
                                        ),
                    loss_function = ProbabilityMSE(),
                    optimizer = tf.optimizers.Adam(learning_rate=0.01),
                    logger = Logger(loss_function_list = [ProbabilityRValue()], sample_freq=100),
                )

    model.train(inputs = inputs,
                targets = targets,
                inputs_val = [inputs],
                targets_val = [targets],
                num_iter = num_iter,
                N = 500,
                verbose=verbose
                )
    
    return model

## Baseline

In [4]:
path = 'data/chaos_exp_data_20251106/baseline_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = loader(f'models/integrable_spam_4_L=5.model')
model_list = loader(f'models/integrable_baseline_model_4_L=5.model')

for i in tqdm(range(5,10)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=5.model')
model_saver(model_list, f'models/integrable_baseline_model_{n}_L=5.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

2025-11-11 01:22:51.730179: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.
2025-11-11 01:22:51.974761: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.


[0.0019936651154444096]


2025-11-11 01:23:53.953377: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.
2025-11-11 01:23:54.205117: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.


...

[-0.0033235639051414356]
[0.5097320609989517]
[0.9184836760997215]
[0.9886505636094567]
[0.994286497350908]
[0.9954309882906399]
[0.9959584756340867]
[0.996280780593749]
[0.99649234014279]
[0.9966470192533399]
[0.9967580090010394]
[0.9968502349725908]
[0.9969157925140741]
[0.9969681703377192]
[0.997009176025399]
[0.9970472636713854]
[0.9970773529000594]
[0.9971080356066478]
[0.9971233461196952]
[0.9971496439242226]
[0.9971591526313313]
[0.997173153259952]
[0.9971805338153997]
[0.9971957588543794]
[0.9972040554378256]
[0.997208668726455]
[0.9972223641541353]
[0.9972264878961414]
[0.9972169723491421]
[0.9972345733720912]
[0.9972246695667828]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.0001200735505788586]
[0.5304452427151891]
[0.9170271416991317]
[0.9887234273895924]
[0.9951013215630106]
[0.9963589079291041]
[0.9969193454238822]
[0.9972460013898842]
[0.9974457724172627]
[0.997590054121742]
[0.9976896365000217]
[0.9977638393093999]
[0.9978203380630162]
[0.9978655269883387]
[0.997905961541784]
[0.9979374027162108]
[0.997959026282115]
[0.9979824867273479]
[0.9980037460758473]
[0.9980181775378894]
[0.9980314825701823]
[0.9980499583987102]
[0.9980551260896005]
[0.9980620205577644]
[0.9980725670113835]
[0.9980822517753473]
[0.998081990784628]
[0.9980897449132722]
[0.9980906083689672]
[0.998093103437256]
[0.9980963803172258]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.001073555462158371]
[0.5366684070078207]
[0.9105501028573573]
[0.9876643921308853]
[0.9948655098205064]
[0.9962105994158059]
[0.9968087229111149]
[0.997157807641419]
[0.9973808031958616]
[0.9975281959533804]
[0.9976367532635982]
[0.9977157455659469]
[0.9977821615316932]
[0.9978270829662875]
[0.9978652474619173]
[0.9979009719886436]
[0.9979235492286064]
[0.9979512900500976]
[0.9979676823857211]
[0.9979827685364824]
[0.9979957615863227]
[0.9980006559724446]
[0.9980151020313422]
[0.9980220686591216]
[0.9980337536492191]
[0.9980415915817478]
[0.9980393106032174]
[0.9980484341574571]
[0.9980485946046964]
[0.9980571383588788]
[0.9980513100744411]


In [5]:
path = 'data/chaos_exp_data_20251106/baseline_L=20_20251019/'
n = 4
d = 2**n
L = 20

spam_list = loader(f'models/integrable_spam_{n}_L=20.model')
model_list = loader(f'models/integrable_baseline_model_{n}_L=20.model')

for i in tqdm(range(5,10)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=20.model')
model_saver(model_list, f'models/integrable_baseline_model_{n}_L=20.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[0.0017088932223893183]
[0.6639414448857497]
[0.9553932999206282]
[0.9836434586875699]
[0.9893653927395533]
[0.991664525085134]
[0.9928812636305384]
[0.9936278835301592]
[0.9941318773013934]
[0.9944978215461108]
[0.994758309713625]
[0.9949664777022367]
[0.9951277542925402]
[0.9952634511167494]
[0.9953702004185655]
[0.9954576394780262]
[0.9955341606104683]
[0.9955915651712456]
[0.9956509990202412]
[0.9956872946929898]
[0.995728378335317]
[0.9957603911373006]
[0.9957904994891624]
[0.9958240416758044]
[0.9958442617832177]
[0.9958549598558959]
[0.9958733708752593]
[0.9958879254778804]
[0.9958960492881973]
[0.9959116434465406]
[0.9959192128459323]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.005162917751268203]
[0.6590441779294864]
[0.9474659388985098]
[0.9783636221460191]
[0.9855048069611368]
[0.9885655237371093]
[0.9901925159447869]
[0.9911635481835326]
[0.9918084747265977]
[0.992265257225508]
[0.9925990029514616]
[0.9928610528456473]
[0.9930571194114878]
[0.9932210223275585]
[0.9933573013236054]
[0.9934630197412294]
[0.9935473496166995]
[0.9936223937122625]
[0.9937005234176268]
[0.9937366152542374]
[0.9937875437458471]
[0.9938291846978746]
[0.993873872198535]
[0.993890044608303]
[0.9939135050306038]
[0.9939392755378796]
[0.993961036497647]
[0.9939720896554443]
[0.9939862398688847]
[0.9940007611363247]
[0.9940257061974457]


...

[-0.003336302991098661]
[0.6880823014775481]
[0.9596123169442315]
[0.9813539605885141]
[0.9872850399896973]
[0.9900757686540068]
[0.9916455626639591]
[0.9926193025883954]
[0.9932750747065113]
[0.9937350005882241]
[0.9940745790162573]
[0.9943316139755405]
[0.9945365909882681]
[0.9946923280933744]
[0.9948269452229866]
[0.9949319381265209]
[0.9950265969493142]
[0.9950993198183012]
[0.9951621329374831]
[0.9952218461117314]
[0.9952712834390394]
[0.9953055623634259]
[0.9953457285439784]
[0.9953786210584744]
[0.9953961924216854]
[0.9954233501306743]
[0.9954461751257596]
[0.9954598580428031]
[0.9954804941827278]
[0.9955000933917901]
[0.9955077812459612]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.008749022742746027]
[0.6585911592314522]
[0.9525659910277945]
[0.9802395564354122]
[0.9867053977477057]
[0.989564541231759]
[0.991152879912229]
[0.9921542438148588]
[0.9928127711944261]
[0.9932861830909372]
[0.9936361559447859]
[0.9938954703942002]
[0.9941140514534876]
[0.9942765128863117]
[0.9944122133005721]
[0.9945206956214583]
[0.9946202961234951]
[0.9946962765004458]
[0.9947629057222607]
[0.9948200350850007]
[0.9948716464472179]
[0.9949087104209183]
[0.9949450892981501]
[0.9949779457755229]
[0.9950076409774287]
[0.9950370624525371]
[0.995054747353828]
[0.9950726478836137]
[0.9950878436637659]
[0.9950950680051025]
[0.9951082775112428]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.00204796540134633]
[0.6947623390757796]
[0.9567342894422873]
[0.9796068351930909]
[0.9856539364442627]
[0.988477183031025]
[0.9900664662018458]
[0.9910498535202207]
[0.9917093893746393]
[0.9921697469150068]
[0.9925142023518887]
[0.9927760596302271]
[0.9929772143713169]
[0.9931508680690989]
[0.9932727911696055]
[0.9933758684388788]
[0.9934766327237747]
[0.9935481871289605]
[0.9936131190946695]
[0.9936660663508016]
[0.9937141798548742]
[0.9937461989829839]
[0.9937906548854007]
[0.9938230880155722]
[0.9938451586358973]
[0.9938525486187267]
[0.9938856828209168]
[0.9938907744632504]
[0.9939107190272909]
[0.9939219138475571]
[0.9939368535298192]


In [3]:
path = 'data/chaos_exp_data_20251106/baseline_L=50_20251019/'
n = 4
d = 2**n
L = 50

spam_list = loader(f'models/integrable_spam_4_L=50.model')
model_list = loader(f'models/integrable_baseline_model_4_L=50.model')

for i in tqdm(range(5,10)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=50.model')
model_saver(model_list, f'models/integrable_baseline_model_{n}_L=50.model')

2025-11-10 16:40:53.328743: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-11-10 16:40:53.328790: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2025-11-10 16:40:53.328798: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-11-10 16:40:53.328803: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-11-10 16:40:53.328809: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: BrigidBrain
2025-11-10 16:40:53.328814: I external/local_xla/xla/stream_executor/cuda/c

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

2025-11-10 16:44:25.882943: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.
2025-11-10 16:44:26.147813: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 3883925504 exceeds 10% of free system memory.


[-0.023025180775350096]


...

[-0.02226866035981323]
[0.7739072781683267]
[0.9032757644254253]
[0.9301708629884249]
[0.9407664881403011]
[0.9464018553560052]
[0.9499183507424791]
[0.9522989980259279]
[0.9540471485399251]
[0.9553204260211204]
[0.9563018702156295]
[0.9570805278762217]
[0.9577389715159742]
[0.9583125771294178]
[0.9586725307050402]
[0.959050229400933]
[0.9593508981831749]
[0.9595620957834279]
[0.959796945494206]
[0.9600302330156558]
[0.9601558960924875]
[0.9602842347005488]
[0.960424886565753]
[0.9605079629797922]
[0.9605875188786899]
[0.9606929345482077]
[0.9607487236457848]
[0.960848762202378]
[0.9608771365453237]
[0.9609258058129391]
[0.9609525506894359]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.01807903076862516]
[0.8177172412614198]
[0.943130510559706]
[0.9657738753340006]
[0.9741590567347119]
[0.9784280545405358]
[0.9809160450793999]
[0.9825204816000515]
[0.983629941047455]
[0.9844404631697123]
[0.9850470961709527]
[0.9855254185849682]
[0.9859023246795097]
[0.986214869234309]
[0.986450566760597]
[0.9866756798275317]
[0.9868787987931531]
[0.9870200792297248]
[0.9871648524691689]
[0.987285992725127]
[0.9873630689022531]
[0.9874671042551253]
[0.9875608847838846]
[0.9876260794951858]
[0.9877089497931015]
[0.9877692657730676]
[0.9877880404136662]
[0.9878414977295709]
[0.9878652446585776]
[0.9879151431163976]
[0.9879424833405468]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.010101543206431307]
[0.8077213679564964]
[0.9346808928099087]
[0.9575631379712816]
[0.9657885006541788]
[0.970049208429818]
[0.972592632621729]
[0.9743343345032343]
[0.9755642414274095]
[0.9764871413119041]
[0.9772039350279186]
[0.9777506679984277]
[0.9782081963519281]
[0.9785811214694861]
[0.9788928181559509]
[0.9791590775089202]
[0.9794021024157251]
[0.9795433765314981]
[0.979749625196795]
[0.9798850827316045]
[0.980009848084159]
[0.9801091494113706]
[0.9802064734827175]
[0.9802886401471403]
[0.9803775403952226]
[0.98042763426556]
[0.980469703122496]
[0.9805362831689738]
[0.980576096086126]
[0.9805962114552255]
[0.9806461840850752]


## Intermediate

In [6]:
path = 'data/chaos_exp_data_20251106/intermediate_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = loader(f'models/integrable_spam_{n}_L=5.model')
model_list = loader(f'models/integrable_intermediate_model_{n}_L=5.model')

for i in tqdm(range(5,10)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=5.model')
model_saver(model_list, f'models/integrable_intermediate_model_{n}_L=5.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[0.002055646445889958]
[0.5591657818195361]
[0.925811131963223]
[0.9885442531521377]
[0.9940625618321253]
[0.9953031264766894]
[0.9959176858666462]
[0.9962873723721521]
[0.9965256995347874]
[0.9966911646279818]
[0.9968084201035634]
[0.9968995037295327]
[0.9969699827231843]
[0.9970250718683212]
[0.9970681785432404]
[0.9971073587689472]
[0.9971371864916098]
[0.9971616716824685]
[0.997181333626254]
[0.9972062551108575]
[0.9972161866705178]
[0.9972381328473462]
[0.9972499092692769]
[0.9972522584436472]
[0.9972621635339645]
[0.9972651725642041]
[0.9972679347619152]
[0.9972792996759291]
[0.9972839790372626]
[0.9972826923859791]
[0.9972842973072696]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.002160749546843199]
[0.5858656474870807]
[0.938111920228494]
[0.9893336798013004]
[0.9942508379751643]
[0.9955733015535776]
[0.9962115240540291]
[0.9965704541276579]
[0.9967954564633548]
[0.9969531119470709]
[0.9970631920632126]
[0.9971456880662154]
[0.9972090088603863]
[0.9972632215617898]
[0.9973068992847648]
[0.9973440833635814]
[0.9973717196736358]
[0.9973995329299394]
[0.9974298322848196]
[0.997440488590728]
[0.9974567399514287]
[0.9974758829575242]
[0.9974846076513361]
[0.9974895805093923]
[0.9975051739500319]
[0.997515764231726]
[0.9975224793762432]
[0.9975279591706836]
[0.9975323061582322]
[0.9975356395564146]
[0.997538706541435]


...

[-0.004097258594536468]
[0.5354023314908496]
[0.9242215352237939]
[0.9872651157294345]
[0.9936593862684714]
[0.9954080491041258]
[0.9961926325301472]
[0.9966284966802157]
[0.9968992887407117]
[0.997086855135107]
[0.9972116230993565]
[0.9973125832692219]
[0.9973916252788108]
[0.997452715844688]
[0.9975027276059527]
[0.997542130558068]
[0.9975774331279877]
[0.9976058854010766]
[0.9976274961526502]
[0.9976476986007751]
[0.997664268087853]
[0.9976814422207988]
[0.997693200657815]
[0.9977012378024344]
[0.9977196980964129]
[0.9977254181442161]
[0.9977320138056803]
[0.9977337317080017]
[0.99773697270131]
[0.9977395081110865]
[0.9977505701620597]


  0%|          | 0/3000 [00:00<?, ?it/s]

[2.449112594449243e-05]
[0.543713726881047]
[0.9237271185293676]
[0.9894251295539811]
[0.9949243635341549]
[0.9961560101761473]
[0.996752632815272]
[0.9971005883149127]
[0.9973216857961353]
[0.9974749214994444]
[0.9975807145843296]
[0.9976622762381546]
[0.997723730969057]
[0.9977757014963377]
[0.9978136470949077]
[0.9978452514991226]
[0.997877057059736]
[0.997897013295269]
[0.9979158402727204]
[0.99793668011661]
[0.9979472933124827]
[0.9979622797163783]
[0.9979775864081025]
[0.9979821647639338]
[0.9979916142762618]
[0.9979951431729709]
[0.9980034709526261]
[0.9980099518699124]
[0.9980101604621094]
[0.9980202602385161]
[0.9980147485976298]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.000999123361409504]
[0.561946869700928]
[0.9232737702876946]
[0.989146945019459]
[0.9950691861858325]
[0.9964477622678383]
[0.997115706980085]
[0.9974969027559861]
[0.9977361595215755]
[0.9978940836334854]
[0.9980090071567237]
[0.998090621383235]
[0.9981511589560477]
[0.9982044667489754]
[0.99824157322246]
[0.9982777323603774]
[0.9982997169456836]
[0.9983262250537429]
[0.9983422224519156]
[0.9983587408874924]
[0.9983726198213722]
[0.9983858121689555]
[0.9983952414141035]
[0.9983999005892058]
[0.9984093879495125]
[0.9984186838189073]
[0.9984262140198708]
[0.998428644460377]
[0.998430305731975]
[0.9984338151053783]
[0.9984364618728492]


In [7]:
path = 'data/chaos_exp_data_20251106/intermediate_L=20_20251019/'
n = 4
d = 2**n
L = 20

spam_list = loader(f'models/integrable_spam_{n}_L=20.model')
model_list = loader(f'models/integrable_intermediate_model_{n}_L=20.model')

for i in tqdm(range(5,10)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)



model_saver(spam_list, f'models/integrable_spam_{n}_L=20.model')
model_saver(model_list, f'models/integrable_intermediate_model_{n}_L=20.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[0.0012024315974772382]
[0.6921689811517473]
[0.9550230622302944]
[0.9818571858029517]
[0.9879141254010659]
[0.9903254085557895]
[0.9915894636043181]
[0.9923880427195626]
[0.992931111875685]
[0.9933218648054886]
[0.9936071688544508]
[0.9938469242161801]
[0.9940307034606343]
[0.9941742145092345]
[0.9943004694919627]
[0.994399588719071]
[0.9944806507382755]
[0.994555423325067]
[0.9946204098489547]
[0.9946663564075179]
[0.9947150716692983]
[0.9947444942203246]
[0.99479539481242]
[0.9948093195134253]
[0.9948361896875368]
[0.9948590079628962]
[0.9948764386927199]
[0.9949047494980917]
[0.9949194382302804]
[0.9949316904828993]
[0.9949413222703665]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.006939659912421847]
[0.683325603179932]
[0.9533469396604871]
[0.9799614597109658]
[0.9865916916951016]
[0.9895057318077086]
[0.9910799987649987]
[0.9920269025156462]
[0.9926663523347632]
[0.9931147053354524]
[0.9934393466215107]
[0.9936959024516533]
[0.9938940468546853]
[0.9940619062915341]
[0.9941944882902677]
[0.9942927905605075]
[0.9943865855745706]
[0.9944574587130119]
[0.9945291264926389]
[0.9945776641884765]
[0.994639918554217]
[0.9946769501434071]
[0.994711545221943]
[0.9947441833351699]
[0.9947651792201059]
[0.9947978567147489]
[0.9948107580860676]
[0.9948306124473845]
[0.9948493369835933]
[0.9948630847282992]
[0.9948793032813079]


...

[-0.007442703615869517]
[0.7511946189370264]
[0.9605791263841277]
[0.9809185555857346]
[0.9862572207250142]
[0.9888013002880598]
[0.9902953529446129]
[0.9912673746804416]
[0.9919294091833204]
[0.9924220091961079]
[0.9927924687646332]
[0.993079408246885]
[0.9933147572892866]
[0.9934962863773253]
[0.9936507599231834]
[0.9937855178614723]
[0.9938919731763858]
[0.9939846793468192]
[0.9940588904444004]
[0.9941276270446797]
[0.9941887232451936]
[0.9942370795172488]
[0.994276902381268]
[0.9943170947051401]
[0.9943562186632251]
[0.9943870661983613]
[0.9944193732353197]
[0.9944407167114538]
[0.9944588510389132]
[0.9944680868071192]
[0.9944849377251156]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.009091424052538155]
[0.7238636792013871]
[0.9402846846503455]
[0.9745921298477078]
[0.9824329152716044]
[0.9856824302064907]
[0.9874781453947611]
[0.9886130709306041]
[0.9893897814423314]
[0.9899531766008601]
[0.9903858531225138]
[0.9907159154062523]
[0.9909895874573761]
[0.9912014440202377]
[0.9913877042725301]
[0.9915313905331741]
[0.9916611179580384]
[0.9917561547905119]
[0.9918554146963973]
[0.9919271127712498]
[0.9919980351759832]
[0.9920684850839934]
[0.9921078855202702]
[0.9921549243347213]
[0.9921926350210668]
[0.992220481633697]
[0.9922560869206496]
[0.9922828749248396]
[0.992306187173044]
[0.9923303022823264]
[0.9923385288489083]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.00024035851264159191]
[0.737386534244383]
[0.9451824687407434]
[0.9726319559418287]
[0.979729034582861]
[0.9828854709591779]
[0.9846821162853268]
[0.9858322966856051]
[0.9866220754123696]
[0.9871963820777592]
[0.9876436706797174]
[0.9879826236342006]
[0.9882515845560844]
[0.9884786617815337]
[0.9886644631253946]
[0.9888012090604955]
[0.9889280928103894]
[0.9890428135796048]
[0.9891242488452711]
[0.989207534819546]
[0.9892728424213524]
[0.9893285182736548]
[0.9893732871606092]
[0.9894043037869856]
[0.9894463671814626]
[0.989481484634857]
[0.9894968758390856]
[0.9895355181421983]
[0.9895377823068937]
[0.9895750561840456]
[0.9895896203032705]


In [8]:
path = 'data/chaos_exp_data_20251106/intermediate_L=50_20251019/'
n = 4
d = 2**n
L = 50

spam_list = loader(f'models/integrable_spam_{n}_L=50.model')
model_list = loader(f'models/integrable_intermediate_model_{n}_L=50.model')

for i in tqdm(range(5,10)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)


model_saver(spam_list, f'models/integrable_spam_{n}_L=50.model')
model_saver(model_list, f'models/integrable_intermediate_model_{n}_L=50.model')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.029920355177103675]
[0.845132575827788]
[0.9398744212868431]
[0.9610544910062067]
[0.9691040530872379]
[0.9732999700876257]
[0.9758543640076678]
[0.9775617814823853]
[0.9787707411668531]
[0.9796594600460421]
[0.9803385173142366]
[0.9808635758740665]
[0.9813140943316512]
[0.9816401784167816]
[0.9819408534304956]
[0.9821888670504453]
[0.9824067111987771]
[0.9825534195820796]
[0.9827228054790264]
[0.9828698056713335]
[0.9829772445548356]
[0.983063438567162]
[0.9831356438533648]
[0.9832521694708398]
[0.9833044394971369]
[0.9833522554630303]
[0.983422680132335]
[0.9834830439384561]
[0.9835313935709438]
[0.9835725273791114]
[0.9836120850507503]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.026803112362165038]
[0.8501233664456225]
[0.9442064676357698]
[0.965030809519994]
[0.9725301304990732]
[0.9764094217217125]
[0.9787941794216413]
[0.9803821000198835]
[0.9815150386742915]
[0.9823461876416156]
[0.9830111369333054]
[0.9835197850148153]
[0.9839346952311204]
[0.9842715846719099]
[0.984558981467104]
[0.98479333161437]
[0.98498503036836]
[0.9851697423564224]
[0.9853339941270681]
[0.9854413895976146]
[0.9855864858421796]
[0.9856743412063691]
[0.9857645363533388]
[0.9858207366583528]
[0.9859283886821308]
[0.9859538170714682]
[0.9860318485415631]
[0.9860645221701043]
[0.986139539990164]
[0.9861437503696022]
[0.9862128742064027]


...

[-0.03303355478976111]
[0.8548715818564275]
[0.9406771662859233]
[0.9627542410549353]
[0.9711863154531574]
[0.9755616718420833]
[0.9781747571247392]
[0.979899528359434]
[0.9811103678472364]
[0.98200618767786]
[0.9826614084606099]
[0.9832146006825845]
[0.9836570982186087]
[0.9840327320881741]
[0.9842833260275342]
[0.9845442456342934]
[0.9847597542676287]
[0.9849167789753427]
[0.9850659441048171]
[0.9851913731635428]
[0.9853127037612354]
[0.9854207235427769]
[0.9855322549416684]
[0.9856085569800255]
[0.9856728303920452]
[0.9857500837743506]
[0.9857864015466875]
[0.9858629156300207]
[0.985917199864174]
[0.9859429398560638]
[0.9859829978185153]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.017213635633839264]
[0.8516610069380572]
[0.9410346051272571]
[0.9637078670363076]
[0.9718415695567623]
[0.9760216102106365]
[0.9785946640435891]
[0.9802915687173922]
[0.981500356994067]
[0.9823857168218609]
[0.9830869272330737]
[0.9836169141119938]
[0.984056559349731]
[0.9843925093417738]
[0.9846978181689245]
[0.9849070145581612]
[0.9851345286338556]
[0.9853019538102297]
[0.9854668941348826]
[0.9855942629270356]
[0.9857115047451864]
[0.9858128860181053]
[0.9858955720214215]
[0.9859857106391574]
[0.9860642889833326]
[0.9861364795608446]
[0.9861805688714393]
[0.9862436401267827]
[0.9862894680746469]
[0.9863215730349468]
[0.9863575032193587]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.008786283641067039]
[0.8457713715023241]
[0.9414731261712931]
[0.9621388261229223]
[0.9697786912814311]
[0.9738363514506431]
[0.9763342478722842]
[0.9780063813545539]
[0.979212020787084]
[0.9801185684458744]
[0.980817151472554]
[0.9813335927178835]
[0.9817666824850902]
[0.9821329541755233]
[0.9824451441224877]
[0.9826904647219357]
[0.9829091746848978]
[0.9830843495781243]
[0.9832413077949131]
[0.9833901354087244]
[0.9835089885994748]
[0.9836216767074141]
[0.9837136054686162]
[0.9838050366375717]
[0.9838591435841124]
[0.9839375644186926]
[0.983987527813886]
[0.984040141740653]
[0.9840822258238008]
[0.9841256478028091]
[0.9841648271156803]


## Baseline T=100

In [5]:
path = 'data/chaos_exp_data_20251106/baseline_L=100_20251121/'
n = 4
d = 2**n
L = 100

spam_list = []
model_list = []

for i in tqdm(range(5)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)


model_saver(spam_list, f'models/integrable_spam_{n}_L=100.model')
model_saver(model_list, f'models/integrable_baseline_model_{n}_L=100.model')

  0%|          | 0/5 [00:00<?, ?it/s]

2025-11-21 13:35:39.042602: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


  0%|          | 0/3000 [00:00<?, ?it/s]

[0.0015350676970692367]
[0.9259402616721516]
[0.9487281400032203]
[0.9593465521698776]
[0.9652187062368576]
[0.9688161951893217]
[0.9712341768033318]
[0.9729027444624218]
[0.9741619127633472]
[0.9751158815055665]
[0.9758643030401306]
[0.9764544985661828]
[0.9769507089076086]
[0.9773655074430337]
[0.9777161459262518]
[0.9780041499820057]
[0.9782329878675808]
[0.9784406115915847]
[0.9786682670853021]
[0.9788050752791797]
[0.9789586775139253]
[0.9790635079792592]
[0.9791937203549158]
[0.9792938561433552]
[0.9793702640209339]
[0.9794716719515012]
[0.9795408054548189]
[0.9796157915334176]
[0.9796811459130113]
[0.9797315792948816]
[0.9797801172612417]


  0%|          | 0/3000 [00:00<?, ?it/s]

...

[-0.007146085362769838]
[0.9398744805333469]
[0.9609278536517559]
[0.9704687130745663]
[0.9755156330060607]
[0.9785212493789648]
[0.980463062243568]
[0.9818045244013205]
[0.9827654462775338]
[0.9834959428018215]
[0.9840403914876422]
[0.984486786829797]
[0.9848236009894016]
[0.9851290626006536]
[0.98536856142978]
[0.9855722497378603]
[0.9857377833034277]
[0.985870160192624]
[0.986022944054273]
[0.9861465785506743]
[0.9862121389602921]
[0.9863329238436254]
[0.9864171878995988]
[0.9864794263991141]
[0.9865332676286047]
[0.9865753674564368]
[0.9866394518932093]
[0.9866894812760307]
[0.9866958137079527]
[0.9867760597811079]
[0.986786913779631]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.005385530344554468]
[0.9345054412075547]
[0.9554457853979521]
[0.9649566168997229]
[0.9702083336406011]
[0.9734877425001076]
[0.9756424959624257]
[0.9771908949979893]
[0.9783175544704529]
[0.9791672190729998]
[0.9798408547198327]
[0.9803937031947857]
[0.9808155779779105]
[0.9811832908016405]
[0.9814953394324416]
[0.9817317875410659]
[0.9819433099140565]
[0.982180207322795]
[0.982314407685191]
[0.9824802947928448]
[0.9825758528022134]
[0.9826906173698012]
[0.9828212632834749]
[0.9828884011849015]
[0.9829940817909838]
[0.9830439265875555]
[0.9831304018822381]
[0.9831706721305448]
[0.98323186473906]
[0.9833033213276798]
[0.9833348910040237]


  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.01946604592611978]
[0.9364007151576004]
[0.9578505097551703]
[0.9677103725145594]
[0.9730280921152673]
[0.9762292918827755]
[0.9783511476345386]
[0.9797983558367631]
[0.9808484268140337]
[0.9816365111681173]
[0.9822793009832684]
[0.9827618595477097]
[0.9831664061636008]
[0.9834975895450792]
[0.98377367695723]
[0.9840011032165406]
[0.984185120032118]
[0.9843865876830784]
[0.9845184871398031]
[0.9846370586359986]
[0.9847923074269214]
[0.9848763890970328]
[0.9849680003404019]
[0.9850480836905874]
[0.9851302278435047]
[0.98518589302378]
[0.9852567080161699]
[0.9852889974059357]
[0.9853321343829823]
[0.9853827434004389]
[0.9854301474511084]


## Non-integrable

In [ ]:
def load_data(filename, n, seed, L):
    with open(filename, 'rb') as f:
        data = pkl.load(f)


    data = marginalize_counts(data, 0)

    targets = counts_to_probs(data)
    targets_spam = targets[:6**n]
    targets_map = targets[6**n:]

    np.random.seed(seed)
    
    circuit_target = nonintegrable_circuit(n+1, L)
    unitary = Operator(circuit_target).data

    inputs_spam, _ = generate_pauliInput_circuits(n)
        
    inputs_map, circuit_list_map = (
                generate_pauli_circuits(n, None, N=5000-6**n)
        )
    
    return inputs_spam, targets_spam, inputs_map, targets_map, unitary

In [ ]:
path = 'data/chaos_exp_data_20251026/nonintegrable_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = []
model_list = []

for i in tqdm(range(10)):
    seed = 42 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)

model_saver(model_list, f'models/nonintegrable_model_{n}_L=5.model')

# Integrable

In [9]:
def load_data(filename, n, seed, L):
    with open(filename, 'rb') as f:
        data = pkl.load(f)


    data = marginalize_counts(data, 0)

    targets = counts_to_probs(data)
    targets_spam = targets[:6**n]
    targets_map = targets[6**n:]

    np.random.seed(seed)
    
    circuit_target = integrable_circuit(n+1, L)
    unitary = Operator(circuit_target).data

    inputs_spam, _ = generate_pauliInput_circuits(n)
        
    inputs_map, circuit_list_map = (
                generate_pauli_circuits(n, None, N=5000-6**n)
        )
    
    return inputs_spam, targets_spam, inputs_map, targets_map, unitary

path = 'data/chaos_exp_data_20251106/integrable_L=5_20251019/'
n = 4
d = 2**n
L = 5

spam_list = loader(f'models/integrable_spam_{n}_L=5.model')
model_list = loader(f'models/integrable_model_{n}_L=5.model')

for i in tqdm(range(5, 8)):
    seed = 52 + i
    inputs_spam, targets_spam, inputs_map, targets_map, unitary = load_data(path + f'seed_{seed}.pkl', n, seed, L)

    tf.random.set_seed(seed)
    spam_model = fit_spam(inputs_spam, targets_spam, verbose=False)
    spam_list.append(spam_model) 

    tf.random.set_seed(seed)
    model = fit_model(inputs_map, 
                     targets_map, 
                     spam_model,
                     num_iter = 3000, 
                     verbose=True)
    model_list.append(model)

model_saver(model_list, f'models/integrable_model_{n}_L=5.model')

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

[-0.00144847496220879]
[0.6105742938896436]
[0.9277620857967447]
[0.9862015729321044]
[0.9939054241149881]
[0.9956871177892688]
[0.9964090298344825]
[0.9968095790168587]
[0.9970656578496543]
[0.9972484811467464]
[0.9973859570079999]
[0.9974885034142521]
[0.9975695554850225]
[0.9976373749529843]
[0.9976928328273681]
[0.9977340417925622]
[0.9977706023868833]
[0.9978053222251418]
[0.9978284890171583]
[0.9978510552937999]
[0.9978754052757408]
[0.9978871722363176]
[0.9979073568253937]
[0.9979165405388292]
[0.997929732283997]
[0.997941564856499]
[0.9979464896616499]
[0.9979565195692426]
[0.9979646287021087]
[0.9979642307456335]
[0.9979788869666665]


  0%|          | 0/3000 [00:00<?, ?it/s]

[0.005351538942743872]
[0.6236787820147153]
[0.9237439196506501]
[0.9844676085713987]
[0.9936716221094176]
[0.9957887050052502]
[0.9965842144438027]
[0.9970007497151953]
[0.9972543443684292]
[0.9974246650354237]
[0.997549815256465]
[0.9976427703563553]
[0.9977154880548675]
[0.9977728983684274]
[0.9978210038396575]
[0.9978638612055214]
[0.9978953068746867]
[0.997921986522529]
[0.9979473774151801]
[0.9979697172287149]
[0.997986826526906]
[0.9980012666144567]
[0.9980113643869588]
[0.9980237325713709]
[0.9980368910130214]
[0.9980475278233155]
[0.9980509186843454]
[0.9980619289332444]
[0.9980667177844874]
[0.9980731274069782]
[0.998076114443445]


  0%|          | 0/3000 [00:00<?, ?it/s]

[0.0032479535726569475]
[0.6239565248946128]
[0.929931994803978]
[0.9863358837633835]
[0.9941625921867926]
[0.9960581942050584]
[0.9968253188707037]
[0.9972300548133209]
[0.9974789900293192]
[0.9976497918551694]
[0.9977702483553808]
[0.9978630850007573]
[0.9979333193534055]
[0.9979906026047312]
[0.998037799962975]
[0.9980761559442062]
[0.9981039883683663]
[0.9981326658022206]
[0.9981573622841227]
[0.998179109081427]
[0.9981947512441948]
[0.9982084608411121]
[0.9982209377532016]
[0.9982359288968947]
[0.9982399379137624]
[0.9982551955702026]
[0.9982618123887079]
[0.9982678974070047]
[0.9982733499686981]
[0.9982771052335365]
[0.9982846609220835]
